
# Deadlock Empire Levels Rewritten in Python

This notebook contains simplified Python versions of classic concurrency problems inspired by Deadlock Empire.

Each section:
- keeps the level name
- shows a minimal Python implementation
- demonstrates the same core issue:
  - race condition
  - deadlock
  - semaphore misuse
  - unsynchronized access


In [ ]:
import threading
import time

# Tutorial

## Tutorial 1: Interface

In [ ]:
flag = True


def thread1():

    time.sleep(0.1)
    print("Thread 1: enter critical section")
    time.sleep(0.1)
    print("Thread 1: exit critical section")

def thread2():
    global flag

    if flag:
        time.sleep(0.1)
        print("Thread 2: enter critical section")
        print("Thread 2: exit critical section")


t1 = threading.Thread(target=thread1)
t2 = threading.Thread(target=thread2)

t1.start()
t2.start()

t1.join()
t2.join()

Thread 1: enter critical section
Thread 2: enter critical section
Thread 2: exit critical section
Thread 1: exit critical section
thread 1: exiting critical_section
thread 1: releasing semaphore
thread 1: trying to wait for semaphore with timeout
thread 1: entering critical_section


## Tutorial 2: Non-Atomic Instructions

In [ ]:
a = 0

def thread1():
  global a

  temp = a + 1
  time.sleep(0.1)
  a = temp

  if (a == 1):
    print("Thread 1: Enter Critical Section")
    time.sleep(0.1)
    print("Thread 1: Exit Critical Section")


def thread2():
  global a

  temp = a + 1
  time.sleep(0.1)
  a = temp

  if (a == 1):
    print("Thread 2: Enter Critical Section")
    time.sleep(0.1)
    print("Thread 2: Exit Critical Section")


t1 = threading.Thread(target=thread1)
t2 = threading.Thread(target=thread2)

t1.start()
t2.start()

t1.join()
t2.join()

Thread 1: Enter Critical Section
Thread 2: Enter Critical Section
Thread 1: Exit Critical Section
Thread 2: Exit Critical Section


# Unsynchronized Code

## Boolean Flags Are Enough For Everyone

Both threads may enter the critical section simultaneously.

In [ ]:
import time
import threading

flag = False
critical_section_busy = False

stop_event = threading.Event()

thread1_passed_guard = threading.Event()
thread2_passed_guard = threading.Event()


def critical_section(name):
    global critical_section_busy

    print(f"{name}: Enter Critical Section")

    if critical_section_busy:
        print("Debug.Assert(false): critical section is taken by both threads")
        stop_event.set()
        return

    critical_section_busy = True
    time.sleep(0.2)
    critical_section_busy = False

    print(f"{name}: Exit Critical Section")


def thread1():
    global flag

    while not stop_event.is_set():
        while flag and not stop_event.is_set():
            pass

        print("Thread 1: passed guard")
        thread1_passed_guard.set()

        thread2_passed_guard.wait()

        flag = True
        print("Thread 1: flag = True")

        critical_section("Thread 1")

        flag = False
        print("Thread 1: flag = False")


def thread2():
    global flag

    while not stop_event.is_set():
        while flag and not stop_event.is_set():
            pass

        print("Thread 2: passed guard")
        thread2_passed_guard.set()

        thread1_passed_guard.wait()

        flag = True
        print("Thread 2: flag = True")

        critical_section("Thread 2")

        flag = False
        print("Thread 2: flag = False")


t1 = threading.Thread(target=thread1)
t2 = threading.Thread(target=thread2)

t1.start()
t2.start()

t1.join()
t2.join()

print("Program stopped")

Thread 1: Enter Critical Section
Thread 1: Exit Critical Section
Thread 1: passed guard
Thread 2: passed guard
Thread 2: flag = True
Thread 2: Enter Critical Section
Thread 1: flag = True
Thread 1: Enter Critical Section
Debug.Assert(false): critical section is taken by both threads
Thread 1: flag = False
Thread 1: Enter Critical Section
cond
Thread 2: Exit Critical Section
Thread 2: flag = False
Program stopped


## Simple Counter

Race condition: both threads read the same value before updating it.

In [ ]:
import time
import threading

counter = 0
critical_section_busy = False

def thread1():
  global critical_section_busy
  global counter
  while True:
    time.sleep(0.1)
    counter += 1
    if counter == 5:
      print("Thread 1: Enter Critical Section")
      if critical_section_busy:
        print("critical section is taken by both threads")
        return
      critical_section_busy = True
      print("Thread 1: Exit Critical Section")
      critical_section_busy = False
    if counter > 5:
      print("infinite loop 1")
      return

t1 = threading.Thread(target=thread1)
t1.start()

def thread2():
  global critical_section_busy
  global counter
  while True:
    counter += 1
    if counter == 3:
      print("Thread 2: Enter Critical Section")
      critical_section_busy = True
      while counter != 5:
        continue
      if critical_section_busy:
        print("critical section is taken by both threads")
        return
      print("Thread 2: Exit Critical Section")
      critical_section_busy = False

t2 = threading.Thread(target=thread2)

t2.start()

t2.join()

Thread 2: Enter Critical Section
Thread 1: Enter Critical Section
critical section is taken by both threads
critical section is taken by both threads


## Confused Counter

In [ ]:
first = 0
second = 0

def thread1():
  global first, second
  temp = first + 1
  time.sleep(0.1)
  first = temp
  temp = second + 1
  second = temp
  if second == 1 and first != 2:
    assert False

def thread2():
  global first, second
  temp = first + 1
  time.sleep(0.1)
  first = temp
  temp = second + 1
  second = temp

t1 = threading.Thread(target=thread1)
t2 = threading.Thread(target=thread2)

t1.start()
t2.start()

t1.join()
t2.join()


Exception in thread Thread-9 (thread1):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_7570/2846232063.py", line 12, in thread1
AssertionError


# Locks

## Insufficient Lock
If more threads are waiting on the same lock, one of them will lock it when it unlocks

In [ ]:
import threading
import time

mutex = threading.Lock()
i = 0
stop_event = threading.Event()


def critical_section(thread_name):
  print(f"{thread_name}: Enter Critical Section")
  time.sleep(0.1)
  print(f"{thread_name}: Exit Critical Section")


def thread1():
  global i

  while not stop_event.is_set():
    with mutex:
      print("Mutex locked by Thread 1")

      i = i + 2
      print("Thread 1: i =", i)

      critical_section("Thread 1")

      if i == 5:
        stop_event.set()
        raise AssertionError()
      print("Mutex unlocked by Thread 1")

      time.sleep(1)


def thread2():
    global i

    while not stop_event.is_set():
        with mutex:
            print("Mutex locked by Thread 2")

            i = i - 1
            print("Thread 2: i =", i)

            critical_section("Thread 2")

            print("Mutex unlocked by Thread 2")

        time.sleep(10)


t1 = threading.Thread(target=thread1)
t2 = threading.Thread(target=thread2)

t2.start()
t1.start()

t1.join()
t2.join()

print("Final i =", i)

Mutex locked by Thread 2
Thread 2: i = -1
Thread 2: Enter Critical Section
Thread 2: Exit Critical Section
Mutex unlocked by Thread 2
Mutex locked by Thread 1
Thread 1: i = 1
Thread 1: Enter Critical Section
Thread 1: Exit Critical Section
Mutex unlocked by Thread 1
Mutex locked by Thread 1
Thread 1: i = 3
Thread 1: Enter Critical Section
Thread 1: Exit Critical Section
Mutex unlocked by Thread 1


Exception in thread Thread-11 (thread1):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_7570/2751493652.py", line 29, in thread1
AssertionError


Mutex locked by Thread 1
Thread 1: i = 5
Thread 1: Enter Critical Section
Thread 1: Exit Critical Section
Final i = 5



## Deadlock

Each thread waits forever for the other lock.

In [ ]:
import threading
import time

mutex1 = threading.Lock()
mutex2 = threading.Lock()

def thread1():
  with mutex1:
    print("Thread 1: Enter the Mutex 1")
    time.sleep(0.1)

    with mutex2:
      print("Thread 1: Enter the Mutex 2")

      print("Thread 1: Enter Critical Section")
      print("Thread 1: Exit Critical Section")

    print("Thread 1: Exit the Mutex 2")
  print("Thread 1: Exit the Mutex 1")

def thread2():
  with mutex2:
    print("Thread 2: Enter the Mutex 2")
    time.sleep(0.1)

    with mutex1:
      print("Thread 2: Enter the Mutex 1")

      print("Thread 2: Enter Critical Section")
      print("Thread 2: Exit Critical Section")
    print("Thread 2: Exit the Mutex 1")
  print("Thread 2: Exit the Mutex 2")



t1 = threading.Thread(target=thread1)
t2 = threading.Thread(target=thread2)

t2.start()
t1.start()

t1.join(timeout=1)
t2.join(timeout=1)


if t1.is_alive() and t2.is_alive():
    print("Deadlock: both threads are waiting for each other")


Thread 2: Enter the Mutex 2
Thread 1: Enter the Mutex 1
Deadlock: both threads are waiting for each other


## A More Complex Thread
Deadlock should be caused

In [ ]:
import threading
import time

mutex1 = threading.RLock()
mutex2 = threading.Lock()
mutex3 = threading.Lock()

flag = False

thread2_locked_mutex1 = threading.Event()
thread1_set_flag_true = threading.Event()
thread2_locked_mutex2 = threading.Event()
thread1_waits_for_mutex2 = threading.Event()


def thread1():
    global flag

    thread2_locked_mutex1.wait()

    got_mutex1 = mutex1.acquire(blocking=False)

    if got_mutex1:
        print("Thread 1: entered IF branch")
    else:
        print("Thread 1: could not lock mutex1, entered ELSE branch")

        mutex2.acquire()
        print("Thread 1: locked mutex2")

        flag = True
        print("Thread 1: flag = True")

        mutex2.release()
        print("Thread 1: released mutex2")

        thread1_set_flag_true.set()

    thread2_locked_mutex2.wait()

    mutex1.acquire()
    print("Thread 1: locked mutex1")

    mutex3.acquire()
    print("Thread 1: locked mutex3")

    mutex1.acquire()
    print("Thread 1: locked mutex1 again")

    mutex1.release()
    print("Thread 1: released mutex1 once")

    print("Thread 1: Enter Critical Section")
    print("Thread 1: Exit Critical Section")

    print("Thread 1: trying to lock mutex2")
    thread1_waits_for_mutex2.set()

    got_mutex2 = mutex2.acquire(timeout=2)

    if not got_mutex2:
        print("Thread 1: cannot lock mutex2, it is held by Thread 2")
        return

    flag = False
    mutex2.release()
    mutex3.release()


def thread2():
    global flag

    if not flag:
        mutex1.acquire()
        print("Thread 2: locked mutex1")

        flag = False
        print("Thread 2: flag = False")

        thread2_locked_mutex1.set()

        thread1_set_flag_true.wait()

        mutex1.release()
        print("Thread 2: released mutex1")

    if flag:
        mutex2.acquire()
        print("Thread 2: locked mutex2")

        thread2_locked_mutex2.set()

        thread1_waits_for_mutex2.wait()

        print("Thread 2: trying to lock mutex1")

        got_mutex1 = mutex1.acquire(timeout=2)

        if not got_mutex1:
            print("Thread 2: cannot lock mutex1, it is held by Thread 1")
            return

        flag = False

        print("Thread 2: Enter Critical Section")
        print("Thread 2: Exit Critical Section")

        mutex1.release()
        mutex2.release()


t2 = threading.Thread(target=thread2)
t1 = threading.Thread(target=thread1)

t2.start()
t1.start()

t1.join()
t2.join()

print("\nResult: deadlock situation reproduced safely")

Thread 2: locked mutex1
Thread 2: flag = False
Thread 1: could not lock mutex1, entered ELSE branch
Thread 1: locked mutex2
Thread 1: flag = True
Thread 1: released mutex2
Thread 2: released mutex1
Thread 2: locked mutex2
Thread 1: locked mutex1
Thread 1: locked mutex3
Thread 1: locked mutex1 again
Thread 1: released mutex1 once
Thread 1: Enter Critical Section
Thread 1: Exit Critical Section
Thread 1: trying to lock mutex2
Thread 2: trying to lock mutex1
Thread 1: cannot lock mutex2, it is held by Thread 2Thread 2: cannot lock mutex1, it is held by Thread 1


Result: deadlock situation reproduced safely


# High-Level Synchronization Primitives

## Manual Reset Event

In [ ]:
import threading
import time

counter = 0

sync = threading.Event()
sync.clear()

def thread0():
    global counter

    while True:
        sync.wait()

        print("Thread 0 reads counter =", counter)
        if counter % 2 == 1:
            assert False


def thread1():
    global counter

    while True:
        sync.clear()
        counter += 1
        print("counter =", counter)
        sync.set()
        time.sleep(0.1)
        counter += 1
        print("counter =", counter)
        sync.set()
        time.sleep(0.5)

t0 = threading.Thread(target=thread0)
t1 = threading.Thread(target=thread1)

t0.start()
t1.start()

t0.join()
t1.join(timeout=2)

Exception in thread Thread-17 (thread0):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_7570/997075975.py", line 17, in thread0
AssertionError


counter = 1
Thread 0 reads counter = 1
counter = 2
counter = 3
counter = 4
counter = 5
counter = 6
counter = 7
counter = 8


## Countdown Event

In [ ]:
import threading
import time

class CountdownEvent:
    def __init__(self, count):
        self.count = count
        self.condition = threading.Condition()

    def signal(self):
        with self.condition:
            self.count -= 1
            print(f"Signal received, waits for {self.count} more signals")

            if self.count <= 0:
                self.condition.notify_all()

    def wait(self):
        with self.condition:
            while self.count > 0:
                self.condition.wait()


progress = 0
event = CountdownEvent(3)

thread1_temp_done = threading.Event()
thread1_progress_done = threading.Event()

def thread0():
    global progress

    thread1_temp_done.wait()

    temp = progress + 20
    print("thread 0: temp = progress + 20 ->", temp)

    thread1_progress_done.wait()
    progress = temp
    print("thread 0: progress =", progress)

    if progress >= 20:
        event.signal()

    print("thread 0 is waiting for event")
    event.wait()
    print("thread 0 is finished")


def thread1():
    global progress

    temp = progress + 30
    print("thread 1: temp = progress + 30 ->", temp)

    thread1_temp_done.set()

    time.sleep(0.3)

    progress = temp
    print("thread 1: progress =", progress)
    thread1_progress_done.set()
    time.sleep(0.2)

    if progress >= 30:
        event.signal()

    progress = progress + 50
    print("thread 1: progress =", progress)

    if progress >= 80:
        event.signal()

    print("thread 1 is waiting for event")
    event.wait()
    print("thread 1 is finished")


t0 = threading.Thread(target=thread0)
t1 = threading.Thread(target=thread1)

t1.start()
t0.start()

t0.join(timeout=3)
t1.join(timeout=3)

if t1.is_alive() and t0.is_alive():
    print("Deadlock is caused")

thread 1: temp = progress + 30 -> 30
thread 0: temp = progress + 20 -> 20
counter = 9
thread 1: progress = 30
thread 0: progress = 20
Signal received, waits for 2 more signals
thread 0 is waiting for event
counter = 10
thread 1: progress = 70
thread 1 is waiting for event
counter = 11
counter = 12
counter = 13
counter = 14
counter = 15
counter = 16
counter = 17
counter = 18
counter = 19
counter = 20
counter = 21
counter = 22
counter = 23
counter = 24
counter = 25
counter = 26
counter = 27
counter = 28
Deadlock is caused


## Countdown Event Revisited

In [ ]:
import os
import threading
import time

class CountdownEvent:
    def __init__(self, count):
        self.count = count
        self.condition = threading.Condition()

    def signal(self):
        with self.condition:
            self.count -= 1
            print(f"Signal received, waits for {self.count} more signals")

            if self.count == 0:
              self.condition.notify_all()

            if self.count < 0:
              print("Countdown is negative!")
              raise RuntimeError("Success")

    def wait(self):
        with self.condition:
            while self.count > 0:
                self.condition.wait()

event = CountdownEvent(3)
progress = 0
thread1_temp = threading.Event()
thread1_progress = threading.Event()

def thread0():
  global progress
  while True:
    thread1_temp.wait()
    temp = progress + 20

    thread1_progress.wait()
    progress = temp
    event.signal()
    if progress == 100:
      print("Environment.Exit(0)")
      return

def thread1():
  global progress
  while True:
    temp = progress + 30
    thread1_temp.set()

    progress = temp
    thread1_progress.set()

    event.signal()
    event.wait()
    if progress == 100:
        print("Environment.Exit(0)")
        return

t0 = threading.Thread(target=thread0)
t1 = threading.Thread(target=thread1)

t1.start()
t0.start()


Signal received, waits for 2 more signals
Signal received, waits for 1 more signals
Signal received, waits for 0 more signals
Signal received, waits for -1 more signals
Countdown is negative!
Signal received, waits for -2 more signals
Countdown is negative!


Exception in thread Thread-21 (thread0):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
Exception in thread Thread-22 (thread1):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_7570/2371664596.py", line 40, in thread0
  File "/tmp/ipykernel_7570/2371664596.py", line 20, in signal
RuntimeError: Success
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_7570/2371664596.py", line 54, in thread1


## The Barrier

In [ ]:
import threading
import time

fireballCharge = 0
charge_lock = threading.Lock()
barrier = threading.Barrier(2)

stop_event = threading.Event()

t0_can_start = threading.Event()
t1_can_start = threading.Event()
t2_can_reset = threading.Event()


def thread0():
    global fireballCharge

    t0_can_start.wait()

    with charge_lock:
        fireballCharge += 1
        print("thread 0: fireballCharge =", fireballCharge)

    print("thread 0: waiting at barrier")
    barrier.wait()
    print("thread 0: passed barrier")

    t1_can_start.set()

    print("thread 0: waiting before check")
    t2_can_reset.wait()

    print("thread 0: checking fireballCharge =", fireballCharge)

    if fireballCharge < 2:
        print("thread 0: Debug.Assert(false)")
        stop_event.set()
        return


def thread1():
    global fireballCharge

    t1_can_start.wait()

    with charge_lock:
        fireballCharge += 1
        print("thread 1: fireballCharge =", fireballCharge)

    print("thread 1: waiting at barrier")
    barrier.wait()
    print("thread 1: passed barrier")


def thread2():
    global fireballCharge

    with charge_lock:
        fireballCharge += 1
        print("thread 2: fireballCharge =", fireballCharge)

    t0_can_start.set()

    print("thread 2: waiting at first barrier")
    barrier.wait()
    print("thread 2: passed first barrier")

    print("thread 2: waiting at second barrier")
    barrier.wait()
    print("thread 2: passed second barrier")

    fireballCharge = 0
    print("thread 2: fireballCharge reset to 0")

    t2_can_reset.set()


t0 = threading.Thread(target=thread0)
t1 = threading.Thread(target=thread1)
t2 = threading.Thread(target=thread2)

t0.start()
t1.start()
t2.start()

t0.join(timeout=5)
t1.join(timeout=5)
t2.join(timeout=5)

if stop_event.is_set():
    print("program stopped after Debug.Assert(false)")

thread 2: fireballCharge = 1
thread 2: waiting at first barrier
thread 0: fireballCharge = 2
thread 0: waiting at barrier
thread 0: passed barrier
thread 0: waiting before check
thread 1: fireballCharge = 3
thread 1: waiting at barrier
thread 2: passed first barrier
thread 2: waiting at second barrier
thread 1: passed barrier
thread 2: passed second barrier
thread 2: fireballCharge reset to 0
thread 0: checking fireballCharge = 0
thread 0: Debug.Assert(false)
program stopped after Debug.Assert(false)


# Semaphores
## Semaphores

In [ ]:
import threading
import time

semaphore = threading.Semaphore(0)

critical_busy = False
stop_event = threading.Event()


def critical_section(name):
    global critical_busy

    print(f"{name}: entering critical_section")

    if critical_busy:
        print("Debug.Assert(false): both threads entered critical_section")
        stop_event.set()
        raise AssertionError("both threads entered critical_section")

    critical_busy = True
    time.sleep(1)
    critical_busy = False

    print(f"{name}: exiting critical_section")


def thread0():
    while not stop_event.is_set():
        print("thread 0: waiting for semaphore")
        semaphore.acquire()

        critical_section("thread 0")

        print("thread 0: releasing semaphore")
        semaphore.release()


def thread1():
    while not stop_event.is_set():
        print("thread 1: trying to wait for semaphore with timeout")

        if semaphore.acquire(timeout=0.5):
            critical_section("thread 1")

            print("thread 1: releasing semaphore")
            semaphore.release()
        else:
            print("thread 1: timeout, releasing semaphore without acquiring")
            semaphore.release()


t0 = threading.Thread(target=thread0)
t1 = threading.Thread(target=thread1)

t0.start()
t1.start()

t0.join(timeout=5)
t1.join(timeout=5)

print("finished")

thread 0: waiting for semaphore
thread 1: trying to wait for semaphore with timeout
thread 1: timeout, releasing semaphore without acquiring
thread 1: trying to wait for semaphore with timeout
thread 1: entering critical_section
thread 1: exiting critical_section
thread 1: releasing semaphore
thread 1: trying to wait for semaphore with timeout
thread 1: entering critical_section
thread 1: exiting critical_section
thread 1: releasing semaphore
thread 1: trying to wait for semaphore with timeout
thread 1: entering critical_section
thread 1: exiting critical_section
thread 1: releasing semaphore
thread 1: trying to wait for semaphore with timeout
thread 1: entering critical_section
thread 1: exiting critical_section
thread 1: releasing semaphore
thread 1: trying to wait for semaphore with timeout
thread 1: entering critical_section
thread 1: exiting critical_section
thread 1: releasing semaphore
thread 1: trying to wait for semaphore with timeout
thread 1: entering critical_section
thread

# Producer-Consumer

In [ ]:
import threading
import time
from collections import deque

semaphore = threading.Semaphore(0)
queue = deque()

stop_event = threading.Event()

producer_released = threading.Event()
consumer_wait_passed = threading.Event()


def thread0():
    producer_released.wait()

    if semaphore.acquire(timeout=0.5):
        print("thread 0: semaphore acquired")

        consumer_wait_passed.set()
        time.sleep(0.2)

        print("thread 0: trying to dequeue")

        try:
            item = queue.popleft()
            print("thread 0: dequeued", item)
        except IndexError:
            print("thread 0: exception, queue is empty")
            stop_event.set()
            raise RuntimeError("queue.Dequeue() from empty queue")
    else:
        print("thread 0: nothing in queue")


def thread1():
    semaphore.release()
    print("thread 1: semaphore released")

    producer_released.set()

    consumer_wait_passed.wait()

    print("thread 1: enqueue Dragon")
    queue.append("Dragon")


t0 = threading.Thread(target=thread0)
t1 = threading.Thread(target=thread1)

t0.start()
t1.start()

t0.join()
t1.join()

thread 1: exiting critical_section
thread 1: releasing semaphore
thread 1: trying to wait for semaphore with timeout
thread 1: entering critical_section
thread 1: semaphore released
thread 0: semaphore acquired
thread 1: enqueue Dragon
thread 0: trying to dequeue
thread 0: dequeued Dragon


## Producer-Consumer (variant)

In [ ]:
import threading
import time

queue = []

first_enqueue_done = threading.Event()
element_lost = threading.Event()
stop_event = threading.Event()


def producer():
    print("producer: queue.Enqueue(new Golem())")
    queue.append("Golem")

    first_enqueue_done.set()

    element_lost.wait()

    print("producer: queue.Enqueue(new Golem())")
    queue.append("Golem")

    stop_event.set()
    print("producer: finished")


def consumer():
    first_enqueue_done.wait()

    if len(queue) > 0:
        print("consumer: queue.Count > 0")

        print("consumer: queue.Dequeue() starts")
        print("consumer: queue enters an inconsistent state")

        lost_item = queue.pop(0)
        print("consumer: queue loses an element:", lost_item)

        element_lost.set()

        time.sleep(0.2)

        print("consumer: queue returns to a consistent state")

    print("consumer: finished")


t_producer = threading.Thread(target=producer)
t_consumer = threading.Thread(target=consumer)

t_producer.start()
t_consumer.start()

t_producer.join()
t_consumer.join()

print("final queue:", queue)
print("processes stopped")

producer: queue.Enqueue(new Golem())
consumer: queue.Count > 0
consumer: queue.Dequeue() starts
consumer: queue enters an inconsistent state
consumer: queue loses an element: Golem
producer: queue.Enqueue(new Golem())
producer: finished
consumer: queue returns to a consistent state
consumer: finished
final queue: ['Golem']
processes stopped


# Condition Variables
## Condition Variables

In [ ]:
import threading
from collections import deque

queue = deque()
condition = threading.Condition()

thread0_waiting = threading.Event()
thread1_waiting = threading.Event()


def thread0():
    with condition:
        print("thread 0: Monitor.Enter(mutex)")

        if len(queue) == 0:
            print("thread 0: queue.Count == 0, Monitor.Wait(mutex)")
            thread0_waiting.set()
            condition.wait()

        print("thread 0: queue.Dequeue()")
        item = queue.popleft()
        print("thread 0: dequeued", item)

        print("thread 0: Monitor.Exit(mutex)")


def thread1():
    with condition:
        print("thread 1: Monitor.Enter(mutex)")

        if len(queue) == 0:
            print("thread 1: queue.Count == 0, Monitor.Wait(mutex)")
            thread1_waiting.set()
            condition.wait()

        print("thread 1: queue.Dequeue()")

        try:
            item = queue.popleft()
            print("thread 1: dequeued", item)
        except IndexError:
            print("InvalidOperationException was thrown")
            print("trying to read from an empty queue")
            raise RuntimeError("trying to read from an empty queue")

        print("thread 1: Monitor.Exit(mutex)")


def thread2():
    thread0_waiting.wait()
    thread1_waiting.wait()

    with condition:
        print("thread 2: Monitor.Enter(mutex)")

        queue.append(42)
        print("thread 2: queue.Enqueue(42)")

        print("thread 2: Monitor.PulseAll(mutex)")
        condition.notify_all()

        print("thread 2: Monitor.Exit(mutex)")


t0 = threading.Thread(target=thread0)
t1 = threading.Thread(target=thread1)
t2 = threading.Thread(target=thread2)

t0.start()
t1.start()
t2.start()

t0.join()
t1.join()
t2.join()

Exception in thread Thread-9 (thread0):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_14266/711038971.py", line 21, in thread0
IndexError: pop from an empty deque


thread 0: Monitor.Enter(mutex)
thread 0: queue.Count == 0, Monitor.Wait(mutex)
thread 1: Monitor.Enter(mutex)
thread 1: queue.Count == 0, Monitor.Wait(mutex)
thread 2: Monitor.Enter(mutex)
thread 2: queue.Enqueue(42)
thread 2: Monitor.PulseAll(mutex)
thread 2: Monitor.Exit(mutex)
thread 1: queue.Dequeue()
thread 1: dequeued 42
thread 1: Monitor.Exit(mutex)
thread 0: queue.Dequeue()


# The Final Stretch


# Dragonfire

In [ ]:
import threading
import time

stop_event = threading.Event()

fireball = threading.Semaphore(0)
firebreathing = threading.Lock()

c = 0
flag = False

thread0_decreased_c = threading.Event()
thread0_in_critical = threading.Event()


def FirebreathingHead():
    global c, flag

    with firebreathing:
        print("Thread 0: Firebreathing locked by Firebreathing Head")

        temp = c - 1
        c = temp
        print("c =", c)

        thread0_decreased_c.set()

        if fireball.acquire(timeout=0.1):
            print("Blast Enemies")
            if fireball.acquire(timeout=0.1):
                if fireball.acquire(timeout=0.1):
                    print("Thread 0: Firebreathing Head enters critical section")

                    if flag:
                        stop_event.set()
                        print("Both threads in critical section!")
                        return
                    else:
                        flag = True

                    thread0_in_critical.set()

                    time.sleep(0.3)

                    print("Thread 0: Firebreathing Head exits critical section")
                    flag = False


def RechargingHead():
    global c, flag

    thread0_decreased_c.wait()

    while c < 2:
        fireball.release()
        c += 1
        print("c =", c)

    thread0_in_critical.wait()

    print("Thread 1: Recharging Head enters critical section")

    if flag:
        stop_event.set()
        print("Both threads in critical section!")
        return
    else:
        flag = True

    print("Thread 1: Recharging Head exits critical section")
    flag = False


t0 = threading.Thread(target=FirebreathingHead)
t1 = threading.Thread(target=RechargingHead)

t0.start()
t1.start()

t0.join()
t1.join()

Thread 0: Firebreathing locked by Firebreathing Head
c = -1
c = 0
c = 1
c = 2
Blast Enemies
Thread 0: Firebreathing Head enters critical section
Thread 1: Recharging Head enters critical section
Both threads in critical section!
Thread 0: Firebreathing Head exits critical section


## Triple Danger

In [ ]:
import threading
from collections import deque

conduit = threading.Lock()
energyBursts = deque(["EnergyBurst", "EnergyBurst", "EnergyBurst"])
event = threading.Event()
stop_event = threading.Event()

def sourcer():
  while not stop_event.is_set():
    conduit.acquire()
    energyBursts.append("Energy Burst")
    conduit.release()

def dragonHeadElectricity():
  while not stop_event.is_set():
    if len(energyBursts) > 0:
      event.wait()
      conduit.acquire()
      energyBursts.popleft()
      stop_event.set()
      print("lightning_bolts(terrifying: true);")
      conduit.release()

def dragonHeadFire():
  while not stop_event.is_set():
    if len(energyBursts) > 0:
      if len(energyBursts) == 1:
        event.set()
      conduit.acquire()
      energyBursts.popleft()
      print("fireball(mighty: true);")
      conduit.release()

t1 = threading.Thread(target=dragonHeadElectricity)
t2 = threading.Thread(target=dragonHeadFire)

t1.start()
t2.start()

time.sleep(3)
stop_event.set()

t1.join()
t2.join()

Exception in thread Thread-19 (dragonHeadElectricity):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_3078/4180382468.py", line 20, in dragonHeadElectricity
IndexError: pop from an empty deque


fireball(mighty: true);
fireball(mighty: true);
fireball(mighty: true);


## Boss Fight

In [3]:
import threading
import time

darkness = 0
evil = 0

fortress = threading.Semaphore(0)
sanctum = threading.Condition()

flag = False
stop_event = threading.Event()

thread0_temp_darkness_ready = threading.Event()
thread1_darkness_done = threading.Event()
thread0_waiting_sanctum = threading.Event()


def thread0():
    global darkness, evil, flag

    while not stop_event.is_set():
        temp = darkness + 1
        thread0_temp_darkness_ready.set()

        thread1_darkness_done.wait()
        darkness = temp

        temp = evil + 1
        evil = temp

        if darkness != 2 and evil != 2:
            if fortress.acquire(timeout=0.5):
                fortress.acquire()

                with sanctum:
                    thread0_waiting_sanctum.set()
                    sanctum.wait()

                    print("Thread 0 enter Critical Section")

                    if flag:
                        print("Both threads in critical section")
                        stop_event.set()
                    else:
                        flag = True

                    time.sleep(0.3)

                    print("Thread 0 exit Critical Section")
                    flag = False


def thread1():
    global darkness, evil, flag

    while not stop_event.is_set():
        thread0_temp_darkness_ready.wait()

        temp = darkness + 1
        darkness = temp
        thread1_darkness_done.set()

        temp = evil + 1
        evil = temp

        fortress.release()
        fortress.release()

        if darkness != 2 and evil == 2:
            thread0_waiting_sanctum.wait()

            with sanctum:
                sanctum.notify()

            print("Thread 1 enter Critical Section")

            if flag:
                print("Both threads in critical section")
                stop_event.set()
            else:
                flag = True

            time.sleep(0.1)

            print("Thread 1 exit Critical Section")
            flag = False

        fortress.release()
        darkness = 0
        evil = 0


t0 = threading.Thread(target=thread0)
t1 = threading.Thread(target=thread1)

t0.start()
t1.start()

t0.join()
t1.join()

Thread 1 enter Critical Section
Thread 0 enter Critical Section
Both threads in critical section
Thread 1 exit Critical Section
Thread 0 exit Critical Section
